In [7]:
import pickle, os, re, hashlib
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any
from pathlib import Path

# Install pypdf if not already installed
!pip install pypdf > /dev/null

PROCESSED_DIR = "/content/drive/MyDrive/SelfImprovingRAG/data/processed"
RAW_DIR = "/content/drive/MyDrive/SelfImprovingRAG/data/raw"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# ── Check if chunks already exist ────────────────────────────────────
parent_path = f"{PROCESSED_DIR}/parent_chunks.pkl"
child_path  = f"{PROCESSED_DIR}/child_chunks.pkl"

if os.path.exists(parent_path) and os.path.exists(child_path):
    with open(parent_path, "rb") as f:
        all_parent_chunks = pickle.load(f)
    with open(child_path, "rb") as f:
        all_child_chunks = pickle.load(f)
    print(f"✅ Loaded from disk — {len(all_parent_chunks)} parents, {len(all_child_chunks)} children")

else:
    print("⚠️  Chunk files not found — rebuilding from raw documents now...")

    # ── Paste Chunk + Document classes ───────────────────────────────
    @dataclass
    class Chunk:
        chunk_id: str
        doc_id: str
        content: str
        metadata: Dict[str, Any] = field(default_factory=dict)
        parent_chunk_id: Optional[str] = None
        chunk_index: int = 0
        total_chunks: int = 0
        def _generate_id(self):
            h = hashlib.md5(f"{self.doc_id}:{self.chunk_index}:{self.content[:100]}".encode()).hexdigest()
            return f"chunk_{h[:12]}"

    @dataclass
    class Document:
        doc_id: str
        content: str
        metadata: Dict[str, Any] = field(default_factory=dict)
        source: str = ""

    # ── Chunker ───────────────────────────────────────────────────────
    class SemanticChunker:
        def __init__(self, child_size=128, parent_size=512, overlap=20):
            self.child_size   = child_size
            self.parent_size  = parent_size
            self.overlap      = overlap

        def _sentences(self, text):
            return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 10]

        def _make_chunks(self, sentences, size, doc_id, prefix):
            chunks, cur, tokens = [], [], 0
            for s in sentences:
                t = len(s) // 4
                if tokens + t > size and cur:
                    c = Chunk(chunk_id="", doc_id=doc_id,
                              content=" ".join(cur), chunk_index=len(chunks),
                              metadata={"prefix": prefix})
                    c.chunk_id = c._generate_id()
                    chunks.append(c)
                    cur = cur[-self.overlap:] + [s]
                    tokens = sum(len(x)//4 for x in cur)
                else:
                    cur.append(s); tokens += t
            if cur:
                c = Chunk(chunk_id="", doc_id=doc_id,
                          content=" ".join(cur), chunk_index=len(chunks),
                          metadata={"prefix": prefix})
                c.chunk_id = c._generate_id()
                chunks.append(c)
            for c in chunks:
                c.total_chunks = len(chunks)
            return chunks

        def chunk_document(self, doc):
            sents   = self._sentences(doc.content)
            parents = self._make_chunks(sents, self.parent_size, doc.doc_id, "parent")
            for p in parents:
                p.metadata.update(doc.metadata)
                p.metadata["source"] = doc.source
            children = []
            for p in parents:
                kids = self._make_chunks(self._sentences(p.content),
                                         self.child_size, doc.doc_id, "child")
                for k in kids:
                    k.parent_chunk_id = p.chunk_id
                    k.metadata.update(p.metadata)
                children.extend(kids)
            return parents, children

    # ── Load raw docs ─────────────────────────────────────────────────
    from pypdf import PdfReader

    documents = []
    raw_path = Path(RAW_DIR)

    if not raw_path.exists() or not any(raw_path.iterdir()):
        print("⚠️  No raw docs found — creating a sample document for demo")
        os.makedirs(RAW_DIR, exist_ok=True)
        sample = (RAW_DIR + "/sample.txt")
        with open(sample, "w") as f:
            f.write("""Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge.\nRetrieval Augmented Generation (RAG) enhances LLMs with external knowledge.
RAG retrieves relevant documents and uses them as context for generation.
BM25 is a sparse retrieval algorithm based on term frequency statistics.
Vector search uses neural embeddings to find semantically similar documents.
Hybrid search combines BM25 and vector search using Reciprocal Rank Fusion.
RRF score equals one divided by the sum of rank position and constant k=60.
Query rewriting transforms vague queries into precise sub-queries.
HyDE generates a hypothetical answer and uses it as the retrieval query.
Faithfulness measures whether answers are grounded in retrieved context.
The reranker uses a cross-encoder to rescore retrieved documents precisely.""")

    for fp in Path(RAW_DIR).rglob("*"):
        if fp.suffix == ".pdf":
            reader = PdfReader(str(fp))
            content = "\n".join(p.extract_text() or "" for p in reader.pages)
            documents.append(Document(doc_id=fp.stem, content=content,
                                      metadata={"type":"pdf"}, source=str(fp)))
        elif fp.suffix in (".txt", ".md"):
            content = fp.read_text(encoding="utf-8", errors="ignore")
            documents.append(Document(doc_id=fp.stem, content=content,
                                      metadata={"type":"txt"}, source=str(fp)))

    print(f"📚 Loaded {len(documents)} documents")

    # ── Chunk everything ───────────────────────────────────────────────
    chunker = SemanticChunker()
    all_parent_chunks, all_child_chunks = [], []
    for doc in documents:
        p, c = chunker.chunk_document(doc)
        all_parent_chunks.extend(p)
        all_child_chunks.extend(c)

    # ── Save to Drive ──────────────────────────────────────────────────
    with open(parent_path, "wb") as f:
        pickle.dump(all_parent_chunks, f)
    with open(child_path, "wb") as f:
        pickle.dump(all_child_chunks, f)

    print(f"✅ Built & saved — {len(all_parent_chunks)} parents, {len(all_child_chunks)} children")

# ── Always show a sample ──────────────────────────────────────────────
if all_child_chunks:
    s = all_child_chunks[0]
    print(f"\n--- Sample Child Chunk ---")
    print(f"ID          : {s.chunk_id}")
    print(f"Parent ID   : {s.parent_chunk_id}")
    print(f"Content     : {s.content[:200]}...")

⚠️  Chunk files not found — rebuilding from raw documents now...
⚠️  No raw docs found — creating a sample document for demo
📚 Loaded 1 documents
✅ Built & saved — 1 parents, 5 children

--- Sample Child Chunk ---
ID          : chunk_1bc5bb4685ab
Parent ID   : chunk_1bc5bb4685ab
Content     : Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge. Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge. RAG retrieves relevant documents and uses them a...


In [8]:
# ⚠️ THIS CELL TAKES 3-5 MINUTES — downloads ~440MB model
from sentence_transformers import SentenceTransformer
import numpy as np

MODEL_NAME = os.environ.get("EMBEDDING_MODEL", "BAAI/bge-base-en-v1.5")
print(f"⏳ Downloading {MODEL_NAME}...")

embed_model = SentenceTransformer(MODEL_NAME)
print(f"✅ Model loaded. Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

# Quick test
test_texts = [
    "The cat sat on the mat",
    "A feline rested on the rug",  # Similar
    "Quantum physics is complex"    # Different
]
test_embeddings = embed_model.encode(test_texts, normalize_embeddings=True)

# Cosine similarity (normalized vectors → dot product = cosine sim)
sim_12 = np.dot(test_embeddings[0], test_embeddings[1])
sim_13 = np.dot(test_embeddings[0], test_embeddings[2])
print(f"\n🔍 Similarity Test:")
print(f"   'cat on mat' vs 'feline on rug' (similar): {sim_12:.4f}")
print(f"   'cat on mat' vs 'quantum physics' (different): {sim_13:.4f}")
print(f"   ✅ Higher score = more similar — works correctly!" if sim_12 > sim_13 else "❌ Something wrong")

⏳ Downloading BAAI/bge-base-en-v1.5...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_19718/344333723.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✅ Model loaded. Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")


✅ Model loaded. Embedding dimension: 768

🔍 Similarity Test:
   'cat on mat' vs 'feline on rug' (similar): 0.7714
   'cat on mat' vs 'quantum physics' (different): 0.3336
   ✅ Higher score = more similar — works correctly!


In [15]:
class Embedder:
    """BGE embedder with correct query prefix handling."""

    def __init__(self):
        self.model = embed_model  # Reuse already-loaded model
        self.dimension = self.model.get_sentence_embedding_dimension()

    def embed_texts(self, texts: list, batch_size=32, normalize=True) -> np.ndarray:
        if not texts:
            return np.array([])
        return self.model.encode(
            texts, batch_size=batch_size,
            normalize_embeddings=normalize,
            convert_to_numpy=True,
            show_progress_bar=len(texts) > 50
        )

    def embed_query(self, query: str) -> np.ndarray:
        # BGE CRITICAL: queries need this prefix for better retrieval
        prefixed = f"Represent this sentence for searching relevant passages: {query}"
        return self.embed_texts([prefixed], normalize=True)[0]

    def embed_document(self, text: str) -> np.ndarray:
        # Documents: NO prefix
        return self.embed_texts([text], normalize=True)[0]

embedder = Embedder()
print(f"✅ Embedder ready. Dimension: {embedder.dimension}")

✅ Embedder ready. Dimension: 768


/tmp/ipykernel_19718/3158861920.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self.model.get_sentence_embedding_dimension()


In [10]:
from tqdm.notebook import tqdm
import numpy as np

print(f"⏳ Embedding {len(all_child_chunks)} child chunks...")
print("   This takes ~2-5 mins depending on document size\n")

# Extract texts
child_texts = [chunk.content for chunk in all_child_chunks]
child_ids = [chunk.chunk_id for chunk in all_child_chunks]

# Embed in batches
BATCH_SIZE = 64
all_embeddings = []

for i in tqdm(range(0, len(child_texts), BATCH_SIZE), desc="Embedding chunks"):
    batch = child_texts[i:i + BATCH_SIZE]
    batch_embeddings = embedder.embed_texts(batch, normalize=True)
    all_embeddings.append(batch_embeddings)

all_embeddings = np.vstack(all_embeddings)
print(f"\n✅ Embeddings shape: {all_embeddings.shape}")
print(f"   {all_embeddings.shape[0]} chunks × {all_embeddings.shape[1]} dimensions")

⏳ Embedding 5 child chunks...
   This takes ~2-5 mins depending on document size



Embedding chunks:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Embeddings shape: (5, 768)
   5 chunks × 768 dimensions


In [11]:
np.save(f"{PROCESSED_DIR}/child_embeddings.npy", all_embeddings)

import json
with open(f"{PROCESSED_DIR}/child_ids.json", "w") as f:
    json.dump(child_ids, f)

print(f"✅ Embeddings saved: child_embeddings.npy ({all_embeddings.nbytes / 1e6:.1f} MB)")
print(f"✅ IDs saved: child_ids.json")

✅ Embeddings saved: child_embeddings.npy (0.0 MB)
✅ IDs saved: child_ids.json


In [13]:
!pip install rank_bm25 > /dev/null
import re
from rank_bm25 import BM25Okapi

STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to",
    "for", "of", "with", "is", "was", "are", "were", "be", "been",
    "being", "have", "has", "had", "do", "does", "did", "will",
    "would", "could", "should", "may", "might", "this", "that", "it"
}

def tokenize(text: str) -> list:
    text = text.lower()
    text = re.sub(r'[^\w\s-]', ' ', text)
    tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

print("⏳ Building BM25 index...")
corpus = [tokenize(text) for text in child_texts]
bm25 = BM25Okapi(corpus)

print(f"✅ BM25 index built")
print(f"   Corpus size: {len(corpus)} documents")

# Quick test
test_query = "retrieval augmented generation"
test_tokens = tokenize(test_query)
scores = bm25.get_scores(test_tokens)
top_indices = scores.argsort()[::-1][:3]

print(f"\n🔍 BM25 Test — Query: '{test_query}'")
for i, idx in enumerate(top_indices):
    print(f"   Rank {i+1} (score={scores[idx]:.3f}): {child_texts[idx][:100]}...")

⏳ Building BM25 index...
✅ BM25 index built
   Corpus size: 5 documents

🔍 BM25 Test — Query: 'retrieval augmented generation'
   Rank 1 (score=-1.954): Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge. Retrieval Augmented Gene...
   Rank 2 (score=-1.997): Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge. Retrieval Augmented Gene...
   Rank 3 (score=-2.036): Retrieval Augmented Generation (RAG) enhances LLMs with external knowledge. Retrieval Augmented Gene...


In [14]:
import pickle

bm25_data = {"bm25": bm25, "chunk_ids": child_ids, "corpus": corpus}
with open(f"{PROCESSED_DIR}/bm25_index.pkl", "wb") as f:
    pickle.dump(bm25_data, f)

print(f"✅ BM25 index saved to {PROCESSED_DIR}/bm25_index.pkl")

✅ BM25 index saved to /content/drive/MyDrive/SelfImprovingRAG/data/processed/bm25_index.pkl
